In [11]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm


DATA_FOLDER = Path("../../data")
INPUT_PATH = DATA_FOLDER / "translated" / "translated_data.csv"

print(f"Input : {INPUT_PATH.resolve()}")

Input : C:\Works\uts-ir\src\data\translated\translated_data.csv


In [2]:
# Load the dataset
df = pd.read_csv(INPUT_PATH)
df.head()


,ID,Title,Abstrak,title_translated,abstract_translated
0,2541-3279-1,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...
1,2541-3279-2,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...
2,2541-3279-3,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...
3,2541-3279-4,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...
4,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...


# Normalizing Text


In [3]:
def normalize_text(text):
    # Handle missing values
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize Unicode without forcing text into ASCII
    text = unicodedata.normalize("NFC", text)

    # Case folding
    text = text.casefold()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)

    # Remove punctuation and special characters,
    # while preserving letters, numbers, and whitespace
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # Replace underscores with spaces
    text = text.replace("_", " ")

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
df["normalized_title"] = df["title_translated"].apply(normalize_text)
df["normalized_abstract"] = df["abstract_translated"].apply(normalize_text)

df[["title_translated", "normalized_title", "abstract_translated", "normalized_abstract"]].sample(20, random_state=42)


,title_translated,normalized_title,abstract_translated,normalized_abstract
4680,analisis kontribusi perpustakaan indonesian vi...,analisis kontribusi perpustakaan indonesian vi...,penelitian ini bertujuan untuk mengetahui kont...,penelitian ini bertujuan untuk mengetahui kont...
1033,tingkat kepuasan pemustaka terhadap mutu pelay...,tingkat kepuasan pemustaka terhadap mutu pelay...,penelitian ini bertujuan untuk mengetahui seja...,penelitian ini bertujuan untuk mengetahui seja...
1044,keaktifan pustakawan dalam pemasyarakatan perp...,keaktifan pustakawan dalam pemasyarakatan perp...,tulisan ini bertujuan untuk memaparkan keaktif...,tulisan ini bertujuan untuk memaparkan keaktif...
3425,dampak rendahnya perpustakaan di masa pandemi ...,dampak rendahnya perpustakaan di masa pandemi ...,masalah yang dihadapi sekolah saat ini adalah ...,masalah yang dihadapi sekolah saat ini adalah ...
4319,representasi profesionalisme pustakawan dalam ...,representasi profesionalisme pustakawan dalam ...,film memiliki peranan penting dalam membentuk ...,film memiliki peranan penting dalam membentuk ...
1905,persepsi pemustaka terhadap penerapan peminjam...,persepsi pemustaka terhadap penerapan peminjam...,penelitian ini bertujuan untuk mengetahui baga...,penelitian ini bertujuan untuk mengetahui baga...
926,pengaruh gerakan literasi sekolah terhadap pen...,pengaruh gerakan literasi sekolah terhadap pen...,penelitian ini bertujuan untuk mengetahui peng...,penelitian ini bertujuan untuk mengetahui peng...
4574,prinsip etika dan tata kelola pengembangan tek...,prinsip etika dan tata kelola pengembangan tek...,keberhasilan tata ti dalam organisasi sangat k...,keberhasilan tata ti dalam organisasi sangat k...
1047,membisniskan layanan perpustakaan menggunakan ...,membisniskan layanan perpustakaan menggunakan ...,perpustakaan sebagai pusat informasi tidak saj...,perpustakaan sebagai pusat informasi tidak saj...
3711,peran tenaga perpustakaan dalam mewujudkan keb...,peran tenaga perpustakaan dalam mewujudkan keb...,penelitian ini berupaya untuk mengetahui peran...,penelitian ini berupaya untuk mengetahui peran...


# Tokenization


In [6]:
df["tokenized_title"] = df["normalized_title"].apply(lambda x: x.split())
df["tokenized_abstract"] = df["normalized_abstract"].apply(lambda x: x.split())

df[["normalized_title", "tokenized_title", "normalized_abstract", "tokenized_abstract"]].sample(20, random_state=42)

,normalized_title,tokenized_title,normalized_abstract,tokenized_abstract
4680,analisis kontribusi perpustakaan indonesian vi...,"[analisis, kontribusi, perpustakaan, indonesia...",penelitian ini bertujuan untuk mengetahui kont...,"[penelitian, ini, bertujuan, untuk, mengetahui..."
1033,tingkat kepuasan pemustaka terhadap mutu pelay...,"[tingkat, kepuasan, pemustaka, terhadap, mutu,...",penelitian ini bertujuan untuk mengetahui seja...,"[penelitian, ini, bertujuan, untuk, mengetahui..."
1044,keaktifan pustakawan dalam pemasyarakatan perp...,"[keaktifan, pustakawan, dalam, pemasyarakatan,...",tulisan ini bertujuan untuk memaparkan keaktif...,"[tulisan, ini, bertujuan, untuk, memaparkan, k..."
3425,dampak rendahnya perpustakaan di masa pandemi ...,"[dampak, rendahnya, perpustakaan, di, masa, pa...",masalah yang dihadapi sekolah saat ini adalah ...,"[masalah, yang, dihadapi, sekolah, saat, ini, ..."
4319,representasi profesionalisme pustakawan dalam ...,"[representasi, profesionalisme, pustakawan, da...",film memiliki peranan penting dalam membentuk ...,"[film, memiliki, peranan, penting, dalam, memb..."
1905,persepsi pemustaka terhadap penerapan peminjam...,"[persepsi, pemustaka, terhadap, penerapan, pem...",penelitian ini bertujuan untuk mengetahui baga...,"[penelitian, ini, bertujuan, untuk, mengetahui..."
926,pengaruh gerakan literasi sekolah terhadap pen...,"[pengaruh, gerakan, literasi, sekolah, terhada...",penelitian ini bertujuan untuk mengetahui peng...,"[penelitian, ini, bertujuan, untuk, mengetahui..."
4574,prinsip etika dan tata kelola pengembangan tek...,"[prinsip, etika, dan, tata, kelola, pengembang...",keberhasilan tata ti dalam organisasi sangat k...,"[keberhasilan, tata, ti, dalam, organisasi, sa..."
1047,membisniskan layanan perpustakaan menggunakan ...,"[membisniskan, layanan, perpustakaan, mengguna...",perpustakaan sebagai pusat informasi tidak saj...,"[perpustakaan, sebagai, pusat, informasi, tida..."
3711,peran tenaga perpustakaan dalam mewujudkan keb...,"[peran, tenaga, perpustakaan, dalam, mewujudka...",penelitian ini berupaya untuk mengetahui peran...,"[penelitian, ini, berupaya, untuk, mengetahui,..."


In [9]:
stopword_factory = StopWordRemoverFactory()
stopwords_id = set(stopword_factory.get_stop_words())

def remove_stopwords(tokens):
    # Token with length greater than 1 and not in stopwords_id
    return [token for token in tokens if token not in stopwords_id and len(token) > 1]

df["tokenized_no_stopwords_title"] = df["tokenized_title"].map(remove_stopwords)
df["tokenized_no_stopwords_abstract"] = df["tokenized_abstract"].map(remove_stopwords)
display(df[["tokenized_title", "tokenized_no_stopwords_title", "tokenized_abstract", "tokenized_no_stopwords_abstract"]].sample(20, random_state=42))
print(f"Jumlah stopword dalam daftar: {len(stopwords_id):,}")

,tokenized_title,tokenized_no_stopwords_title,tokenized_abstract,tokenized_no_stopwords_abstract
4680,"[analisis, kontribusi, perpustakaan, indonesia...","[analisis, kontribusi, perpustakaan, indonesia...","[penelitian, ini, bertujuan, untuk, mengetahui...","[penelitian, bertujuan, mengetahui, kontribusi..."
1033,"[tingkat, kepuasan, pemustaka, terhadap, mutu,...","[tingkat, kepuasan, pemustaka, mutu, pelayanan...","[penelitian, ini, bertujuan, untuk, mengetahui...","[penelitian, bertujuan, mengetahui, sejauhmana..."
1044,"[keaktifan, pustakawan, dalam, pemasyarakatan,...","[keaktifan, pustakawan, pemasyarakatan, perpus...","[tulisan, ini, bertujuan, untuk, memaparkan, k...","[tulisan, bertujuan, memaparkan, keaktifan, pu..."
3425,"[dampak, rendahnya, perpustakaan, di, masa, pa...","[dampak, rendahnya, perpustakaan, masa, pandem...","[masalah, yang, dihadapi, sekolah, saat, ini, ...","[masalah, dihadapi, sekolah, masa, pandemic, c..."
4319,"[representasi, profesionalisme, pustakawan, da...","[representasi, profesionalisme, pustakawan, fi...","[film, memiliki, peranan, penting, dalam, memb...","[film, memiliki, peranan, penting, membentuk, ..."
1905,"[persepsi, pemustaka, terhadap, penerapan, pem...","[persepsi, pemustaka, penerapan, peminjaman, m...","[penelitian, ini, bertujuan, untuk, mengetahui...","[penelitian, bertujuan, mengetahui, bagaimana,..."
926,"[pengaruh, gerakan, literasi, sekolah, terhada...","[pengaruh, gerakan, literasi, sekolah, peningk...","[penelitian, ini, bertujuan, untuk, mengetahui...","[penelitian, bertujuan, mengetahui, pengaruh, ..."
4574,"[prinsip, etika, dan, tata, kelola, pengembang...","[prinsip, etika, tata, kelola, pengembangan, t...","[keberhasilan, tata, ti, dalam, organisasi, sa...","[keberhasilan, tata, ti, organisasi, sangat, k..."
1047,"[membisniskan, layanan, perpustakaan, mengguna...","[membisniskan, layanan, perpustakaan, mengguna...","[perpustakaan, sebagai, pusat, informasi, tida...","[perpustakaan, pusat, informasi, mengemban, fu..."
3711,"[peran, tenaga, perpustakaan, dalam, mewujudka...","[peran, tenaga, perpustakaan, mewujudkan, kebe...","[penelitian, ini, berupaya, untuk, mengetahui,...","[penelitian, berupaya, mengetahui, peran, perp..."


Jumlah stopword dalam daftar: 123


# Stemming


In [12]:
stemmer = StemmerFactory().create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

# Stem token pada judul
df["tokens_stemmed_title"] = [
    stem_tokens(tokens)
    for tokens in tqdm(
        df["tokenized_no_stopwords_title"],
        desc="Stemming titles"
    )
]

# Stem token pada abstrak
df["tokens_stemmed_abstract"] = [
    stem_tokens(tokens)
    for tokens in tqdm(
        df["tokenized_no_stopwords_abstract"],
        desc="Stemming abstracts"
    )
]

df["text_stemmed_title"] = (
    df["tokens_stemmed_title"].apply(" ".join)
)

df["text_stemmed_abstract"] = (
    df["tokens_stemmed_abstract"].apply(" ".join)
)


Stemming abstracts: 100%|██████████| 5890/5890 [19:24<00:00,  5.06it/s]  


KeyError: "['tokens_no_stopwords_title', 'tokens_no_stopwords_abstract'] not in index"

In [14]:
# Periksa hasilnya
df[[
    "tokenized_no_stopwords_title",
    "tokens_stemmed_title",
    "text_stemmed_title",
    "tokenized_no_stopwords_abstract",
    "tokens_stemmed_abstract",
    "text_stemmed_abstract"
]].sample(min(20, len(df)), random_state=42)

,tokenized_no_stopwords_title,tokens_stemmed_title,text_stemmed_title,tokenized_no_stopwords_abstract,tokens_stemmed_abstract,text_stemmed_abstract
4680,"[analisis, kontribusi, perpustakaan, indonesia...","[analisis, kontribusi, pustaka, indonesian, vi...",analisis kontribusi pustaka indonesian visual ...,"[penelitian, bertujuan, mengetahui, kontribusi...","[teliti, tuju, tahu, kontribusi, pustaka, indo...",teliti tuju tahu kontribusi pustaka indonesian...
1033,"[tingkat, kepuasan, pemustaka, mutu, pelayanan...","[tingkat, puas, mustaka, mutu, layan, pustaka,...",tingkat puas mustaka mutu layan pustaka bas si...,"[penelitian, bertujuan, mengetahui, sejauhmana...","[teliti, tuju, tahu, sejauhmana, tingkat, puas...",teliti tuju tahu sejauhmana tingkat puas musta...
1044,"[keaktifan, pustakawan, pemasyarakatan, perpus...","[aktif, pustakawan, masyarakat, perpusdokinfo,...",aktif pustakawan masyarakat perpusdokinfo ting...,"[tulisan, bertujuan, memaparkan, keaktifan, pu...","[tulis, tuju, papar, aktif, pustakawan, masyar...",tulis tuju papar aktif pustakawan masyarakat p...
3425,"[dampak, rendahnya, perpustakaan, masa, pandem...","[dampak, rendah, pustaka, masa, pandemi, covid...",dampak rendah pustaka masa pandemi covid 19 si...,"[masalah, dihadapi, sekolah, masa, pandemic, c...","[masalah, hadap, sekolah, masa, pandemic, covi...",masalah hadap sekolah masa pandemic covid 19 h...
4319,"[representasi, profesionalisme, pustakawan, fi...","[representasi, profesionalisme, pustakawan, fi...",representasi profesionalisme pustakawan film t...,"[film, memiliki, peranan, penting, membentuk, ...","[film, milik, peran, penting, bentuk, persepsi...",film milik peran penting bentuk persepsi masya...
1905,"[persepsi, pemustaka, penerapan, peminjaman, m...","[persepsi, mustaka, terap, pinjam, mandiri, pu...",persepsi mustaka terap pinjam mandiri pustaka ...,"[penelitian, bertujuan, mengetahui, bagaimana,...","[teliti, tuju, tahu, bagaimana, persepsi, must...",teliti tuju tahu bagaimana persepsi mustaka te...
926,"[pengaruh, gerakan, literasi, sekolah, peningk...","[pengaruh, gera, literasi, sekolah, tingkat, l...",pengaruh gera literasi sekolah tingkat literas...,"[penelitian, bertujuan, mengetahui, pengaruh, ...","[teliti, tuju, tahu, pengaruh, gera, literasi,...",teliti tuju tahu pengaruh gera literasi sekola...
4574,"[prinsip, etika, tata, kelola, pengembangan, t...","[prinsip, etika, tata, kelola, kembang, teknol...",prinsip etika tata kelola kembang teknologi in...,"[keberhasilan, tata, ti, organisasi, sangat, k...","[hasil, tata, ti, organisasi, sangat, konsiste...",hasil tata ti organisasi sangat konsisten tera...
1047,"[membisniskan, layanan, perpustakaan, mengguna...","[bisnis, layan, pustaka, guna, commerce]",bisnis layan pustaka guna commerce,"[perpustakaan, pusat, informasi, mengemban, fu...","[pustaka, pusat, informasi, emban, fungsi, sos...",pustaka pusat informasi emban fungsi sosial sa...
3711,"[peran, tenaga, perpustakaan, mewujudkan, kebe...","[peran, tenaga, pustaka, wujud, hasil, gera, l...",peran tenaga pustaka wujud hasil gera literasi...,"[penelitian, berupaya, mengetahui, peran, perp...","[teliti, upaya, tahu, peran, pustaka, sekolah,...",teliti upaya tahu peran pustaka sekolah tenaga...


In [20]:
# Export the processed DataFrame to a new CSV file
df_processed = pd.DataFrame({
    "id": df["ID"],
    "title": df["Title"],
    "abstract": df["Abstrak"],
    "title_translated": df["title_translated"],
    "abstract_translated": df["abstract_translated"],
    "processed_token_title": df["tokens_stemmed_title"],
    "processed_token_abstract": df["tokens_stemmed_abstract"],
    "processed_title": df["text_stemmed_title"],
    "processed_abstract": df["text_stemmed_abstract"],
})

df_processed.head()

,id,title,abstract,title_translated,abstract_translated,processed_token_title,processed_token_abstract,processed_title,processed_abstract
0,2541-3279-1,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,"[hubung, promosi, pustaka, lalu, majalah, dind...","[teliti, dilatarbelakangi, kembang, jaman, mak...",hubung promosi pustaka lalu majalah dinding mi...,teliti dilatarbelakangi kembang jaman makin ma...
1,2541-3279-2,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,"[hubung, guna, search, features, journals, pen...","[teliti, bahas, hubung, guna, search, features...",hubung guna search features journals penuh but...,teliti bahas hubung guna search features journ...
2,2541-3279-3,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,"[hubung, sedia, fasilitas, pustaka, minat, kun...","[teliti, bahas, hubung, sedia, fasilitas, pust...",hubung sedia fasilitas pustaka minat kunjung s...,teliti bahas hubung sedia fasilitas pustaka mi...
3,2541-3279-4,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,"[pengaruh, kerja, layan, prima, center, of, in...","[masalah, jadi, kaji, teliti, kena, layan, pri...",pengaruh kerja layan prima center of informati...,masalah jadi kaji teliti kena layan prima inti...
4,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,"[kontribusi, literasi, informasi, mahasiswa, p...","[fenomena, pesat, kembang, ilmu, tahu, teknolo...",kontribusi literasi informasi mahasiswa proses...,fenomena pesat kembang ilmu tahu teknologi inf...


In [21]:
OUTPUT_FOLDER_PATH = DATA_FOLDER / "processed"
OUTPUT_FOLDER_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_FOLDER_PATH / "processed_data.csv"
df_processed.to_csv(OUTPUT_PATH, index=False)

print(f"Processed data saved to: {OUTPUT_PATH.resolve()}")

Processed data saved to: C:\Works\uts-ir\src\data\processed\processed_data.csv
